# scOntoFM demo: preprocessing → tokenization → inference

This notebook walks through the full scOntoFM inference pipeline on a small example
dataset (`examples/data/demo_pancreas.h5ad`: 2,000 human pancreatic islet cells from
4 donors, CEL-seq2; subsampled from the CELLxGENE-formatted Muraro et al. 2016 pancreas atlas).

1. **Preprocessing**: raw counts → cell/gene filtering → library-size normalization (no log transform)
2. **Tokenization**: median-normalized rank-value encoding, `[<cls>] + top-2,000 genes + [<eos>]`
3. **Inference**: 256-d cell embeddings from the pretrained scOntoFM encoder (`<cls>` token by default)
4. **Quick look**: UMAP + clustering agreement with the annotated cell types

Runs on GPU or CPU (the model has ~8.9M parameters; CPU takes a few minutes for 2,000 cells).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import scanpy as sc
import torch

from scOntoFM.paths import GENE_MEDIAN_PATH, MODEL_DIR, TOKEN_DICT_PATH
from scOntoFM.preprocessing import preprocess_single_h5ad
from scOntoFM.tokenization import (load_gene_median_dict, load_token_dictionary,
                                   shared_genes, tokenize_adata)
from scOntoFM.data import build_hf_dataset
from scOntoFM.model import load_model, embed_cells

DEMO_H5AD = REPO_ROOT / "examples" / "data" / "demo_pancreas.h5ad"
WORK_DIR = REPO_ROOT / "examples" / "demo_output"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Input data

scOntoFM expects **raw counts** (in `adata.raw.X` or `adata.X`) and **Ensembl gene IDs** as
`var_names` (the CELLxGENE convention). If your data uses gene symbols, map them to Ensembl IDs first.

In [ ]:
adata_raw = sc.read_h5ad(DEMO_H5AD)
print(adata_raw)
print("example var_names:", adata_raw.var_names[:5].tolist())
adata_raw.obs["cell_type"].value_counts()

## 2. Preprocessing

`preprocess_single_h5ad` keeps `is_primary_data == True` cells (if the column exists), filters
cells/genes, normalizes every cell to 10,000 total counts and writes a new `.h5ad`
(cached: an existing output file is reused). The embeddings produced later are row-aligned with this
preprocessed file.

In [ ]:
preprocessed_path = preprocess_single_h5ad(
    str(DEMO_H5AD),
    min_genes_per_cell=0,      # inference default; the pretraining corpus used 200
    min_cells_per_gene=0,      # inference default; the pretraining corpus used 5
    target_sum=1e4,
    out_dir=str(WORK_DIR / "preprocessed"),
)
adata = sc.read_h5ad(preprocessed_path)
print(adata)
print("total counts per cell (first 5):", np.asarray(adata.X[:5].sum(axis=1)).ravel())

## 3. Tokenization

* `token_dictionary.pkl`: 19,941 tokens: `<pad>=0, <mask>=1, <cls>=2, <eos>=3` + 19,937 Ensembl IDs
* `gene_medians.pkl`: non-zero median expression of each gene over the pretraining corpus

Each gene's normalized value is divided by its corpus median, the expressed genes are ranked in
descending order, and the top 2,000 are kept. Dividing by the median down-weights ubiquitously
high (housekeeping) genes and up-weights genes that are specifically high in a cell.

In [ ]:
token_dict = load_token_dictionary(TOKEN_DICT_PATH)
gene_medians = load_gene_median_dict(GENE_MEDIAN_PATH)
id2token = {v: k for k, v in token_dict.items()}
print("vocab size:", len(token_dict), "| special tokens:", {k: token_dict[k] for k in ["<pad>", "<mask>", "<cls>", "<eos>"]})

genes = shared_genes(adata, gene_medians)
print(f"{len(genes)} / {adata.n_vars} genes of the dataset are in the scOntoFM vocabulary")

tokens = tokenize_adata(adata, genes, gene_medians, token_dict, top_k=2000)
lengths = np.array([len(t) for t in tokens])
print(f"sequence length: min {lengths.min()}, median {int(np.median(lengths))}, max {lengths.max()}")

# decode the first cell's top-ranked genes
symbol = adata_raw.var["feature_name"].astype(str).to_dict() if "feature_name" in adata_raw.var else {}
first = tokens[0]
print("cell 0:", adata.obs["cell_type"].iloc[0])
print("token ids:", first[:12], "...")
print("top genes:", [symbol.get(id2token[t], id2token[t]) for t in first[1:11]])

The same tokenization wrapped as a HuggingFace `Dataset` (what the model consumes). This is
equivalent to `python scripts/tokenize_h5ad.py --format hf`.

In [ ]:
dataset, token_dict = build_hf_dataset(preprocessed_path, TOKEN_DICT_PATH, GENE_MEDIAN_PATH, top_k=2000)
assert dataset["input_ids"] == tokens
dataset

## 4. Inference

The released weights (`resources/weights/scOntoFM`) are a HuggingFace `BertForMaskedLM` checkpoint
(6 layers, hidden size 256, 4 heads). The default cell embedding is the last-layer `<cls>` state,
which is the representation shaped by the ontology-alignment and triplet objectives during pretraining.
`pooling="mean"` gives the attention-masked mean over all tokens instead.

In [ ]:
model, device = load_model(MODEL_DIR)
print("device:", device, "| parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

cell_emb, _ = embed_cells(model, dataset, token_dict, device, pooling="cls", batch_size=32, num_workers=0)
cell_emb = cell_emb.numpy()
print("cell embeddings:", cell_emb.shape)
np.save(WORK_DIR / "demo_pancreas_embeddings.npy", cell_emb)

## 5. Visualize and evaluate

UMAP of the scOntoFM embedding, colored by cell type and donor (the batch variable), plus the agreement
between k-means clusters and the annotations.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

adata.obsm["scOntoFM"] = cell_emb
sc.pp.neighbors(adata, use_rep="scOntoFM", n_neighbors=10)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cell_type", "donor_id"], frameon=False, wspace=0.6)

k = adata.obs["cell_type"].nunique()
pred = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(cell_emb)
print(f"k-means (k={k}) vs cell_type: ARI = {adjusted_rand_score(adata.obs['cell_type'], pred):.3f}, "
      f"NMI = {normalized_mutual_info_score(adata.obs['cell_type'], pred):.3f}")

## 6. Command-line equivalent

The whole notebook corresponds to a single call (see `examples/run_demo.sh`):

```bash
python scripts/run_inference.py \
    --input_h5ad examples/data/demo_pancreas.h5ad \
    --output_dir examples/demo_output --run_id scOntoFM \
    --color_obs cell_type donor_id --save_data
```

Outputs: `examples/demo_output/scOntoFM/demo_pancreas/demo_pancreas_embeddings.npy` (+ tokenized
dataset, annotated `.h5ad`, UMAP PDFs). Add `--pooling mean` for mean pooling and
`--save_gene_embeddings` for contextual gene embeddings (used by
`downstream/3_gene_function/GO_embedding_contextualize.py`). These embedding files are the input
of all downstream analyses in `downstream/`.